# Description

In this notebook, I will train the Tokenizer (BPE) from the dataset

In [ ]:
import os 
import numpy as np
import pandas as pd

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
from tokenizers.normalizers import Sequence, Lowercase, NFD, StripAccents
from tokenizers.pre_tokenizers import Whitespace
from pathlib import Path
import tempfile

In [ ]:
PATH_FILE_DATA = os.path.join(os.getcwd(), 'dataset', 'processed', 'processed_data.csv')

# 1. Read data

In [ ]:
df = pd.read_csv(PATH_FILE_DATA)
print(f"Number of records in processed data: {len(df)}")

In [ ]:
list_python_function = df['method_code'].tolist()
print(f"Number of python functions: {len(list_python_function)}")

In [ ]:
idx = np.random.randint(0, len(list_python_function))
print("Example of python function:")
print(list_python_function[idx])

Before training tokenizer, we estimate a vocab size

In [ ]:
def estimate_vocab_size(functions):
    total_chars = sum(len(f) for f in functions)
    # simple heuristic: 1 token per 4–5 chars, capped at 64k
    est = min(max(total_chars // 5, 2000), 64000)
    return int(round(est, -2))  

vocab_size = estimate_vocab_size(list_python_function)
print("Estimated vocab size:", vocab_size)

# 2. Train BPE

In [ ]:
def train_bpe_tokenizer(functions, vocab_size=64_000, save_path="python_tokenizer.json"):
    """
    Train a Byte-Pair Encoding (BPE) tokenizer on a list of Python functions.
    """
    # 1. Create a temporary file with one function per line
    with tempfile.NamedTemporaryFile("w+", delete=False) as tmp:
        tmp_path = Path(tmp.name)
        for func in functions:
            func = func.replace("\n", " ")  # flatten to single line
            tmp.write(func + "\n")
    
    # 2. Initialize a BPE tokenizer
    tokenizer = Tokenizer(models.BPE())
    tokenizer.normalizer = Sequence([NFD(), Lowercase(), StripAccents()])
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()  # good for code data

    # 3. Define trainer
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        show_progress=True,
        special_tokens=["<pad>", "<unk>", "<s>", "</s>", "<mask>"],
    )

    tokenizer.train([str(tmp_path)], trainer)

    tokenizer.save(save_path)
    print(f"[DONE] Trained tokenizer saved to: {save_path}")
    return tokenizer

In [ ]:
output_tokenizer_path = os.path.join(os.getcwd(), 'python_tokenizer.json')
vocab_size = estimate_vocab_size(list_python_function)
print("Estimated vocab size:", vocab_size)

In [ ]:
tokenizer = train_bpe_tokenizer(list_python_function, vocab_size=vocab_size, save_path=output_tokenizer_path)

# 3. Test Tokenizer

In [ ]:
encoded = tokenizer.encode("def add(a, b): return a + b")
print("Tokens:", encoded.tokens, "\n")
print("IDs:", encoded.ids, "\n")
print("Decoded:", tokenizer.decode(encoded.ids), "\n")